In [6]:
import json
import random
import math
from pathlib import Path
from copy import deepcopy


def load_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_span(span):
    """
    Ensures spans follow:
    {
      "span": "...",
      "start": ...,
      "end": ...
    }
    """
    span = deepcopy(span)

    if "span" not in span and "text" in span:
        span["span"] = span.pop("text")

    return span


def normalize_example(example):
    """
    Keeps only the fields needed for manual audit:
    spans, document, reason.
    """
    example = deepcopy(example)

    spans = example.get("spans", [])
    spans = [normalize_span(s) for s in spans]

    return {
        "spans": spans,
        "document": example.get("document", ""),
        "reason": example.get("reason", "")
    }


def group_silver_data(data):
    """
    Supports either:

    1. Category-grouped format:
       {
         "Unsupported claim": [...],
         "Format": [...]
       }

    2. Flat list format:
       [
         {"category": "Unsupported claim", "spans": ..., ...}
       ]
    """
    grouped = {}

    if isinstance(data, dict):
        for category, examples in data.items():
            grouped[category] = [normalize_example(ex) for ex in examples]

    elif isinstance(data, list):
        for ex in data:
            category = ex.get("category")
            if category is None:
                raise ValueError("Flat list format requires each example to have a 'category' field.")

            grouped.setdefault(category, []).append(normalize_example(ex))

    else:
        raise ValueError("Unsupported JSON format. Expected dict or list.")

    return grouped


def sample_silver_json_by_category(
    input_path,
    output_path,
    n_per_category=None,
    percent_per_category=None,
    seed=13
):
    """
    Creates a stratified JSON sample grouped by category.

    Use either:
      - n_per_category=25
      - percent_per_category=0.10

    Output format:
    {
      "Lacks synthesis": [
        {
          "spans": [...],
          "document": "...",
          "reason": "..."
        }
      ]
    }
    """
    if n_per_category is None and percent_per_category is None:
        raise ValueError("Specify either n_per_category or percent_per_category.")

    random.seed(seed)

    data = load_json(input_path)
    grouped = group_silver_data(data)

    sampled_grouped = {}

    for category, examples in grouped.items():
        examples = deepcopy(examples)

        if percent_per_category is not None:
            n = max(1, math.ceil(len(examples) * percent_per_category))
        else:
            n = n_per_category

        n = min(n, len(examples))
        sampled_grouped[category] = random.sample(examples, n)

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(sampled_grouped, f, indent=2, ensure_ascii=False)

    print(f"Saved sampled JSON to: {output_path}")
    for category, examples in sampled_grouped.items():
        print(f"{category}: {len(examples)} examples")

    return sampled_grouped

In [7]:
sampled = sample_silver_json_by_category(
    input_path="samples/synthetic_samples_complete.json",
    output_path="silver_audit_sample.json",
    n_per_category=25,
    seed=13
)

Saved sampled JSON to: silver_audit_sample.json
Unsupported claim: 25 examples
Lacks synthesis: 25 examples
Format: 25 examples
Coherence: 25 examples


In [1]:
import json
import random
from pathlib import Path
import pandas as pd


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def find_span_offsets(document, span_text, start=None, end=None):
    """
    Use provided start/end if available; otherwise find span text in document.
    """
    if start is not None and end is not None:
        return int(start), int(end)

    idx = document.find(span_text)
    if idx == -1:
        return None, None

    return idx, idx + len(span_text)


def make_context(document, start, end, window=400):
    """
    Create a compact context string with the target span marked.
    """
    if start is None or end is None:
        return document[:1000]

    left = max(0, start - window)
    right = min(len(document), end + window)

    return (
        document[left:start]
        + " [[SPAN_START]] "
        + document[start:end]
        + " [[SPAN_END]] "
        + document[end:right]
    )


def flatten_synthetic_json(data):
    rows = []

    for category, examples in data.items():
        for ex_idx, ex in enumerate(examples):
            document = ex.get("document", "")
            reason = ex.get("reason", "")

            for span_idx, span in enumerate(ex.get("spans", [])):
                span_text = span.get("span", span.get("text", ""))
                start = span.get("start", None)
                end = span.get("end", None)

                start, end = find_span_offsets(
                    document=document,
                    span_text=span_text,
                    start=start,
                    end=end,
                )

                rows.append({
                    "category": category,
                    "example_id": f"{category}_{ex_idx}",
                    "span_id": f"{category}_{ex_idx}_{span_idx}",
                    "span_text": span_text,
                    "context": make_context(document, start, end),
                    "reason": reason,
                    "category_valid": "",
                    "boundary_plausible": "",
                    "context_plausible": "",
                    "notes": "",
                })

    return pd.DataFrame(rows)


def make_audit_sample(
    input_json_path,
    output_csv_path="synthetic_span_audit.csv",
    n_per_category=25,
    seed=13,
):
    random.seed(seed)

    data = load_json(input_json_path)
    df = flatten_synthetic_json(data)

    sampled = []
    for category, group in df.groupby("category"):
        n = min(n_per_category, len(group))
        sampled.append(group.sample(n=n, random_state=seed))

    audit_df = pd.concat(sampled, ignore_index=True)
    audit_df.to_csv(output_csv_path, index=False)

    print(f"Saved {len(audit_df)} audit rows to {output_csv_path}")
    print(audit_df["category"].value_counts())

    return audit_df

In [2]:
audit_df = make_audit_sample(
    input_json_path="silver_audit_sample.json",
    output_csv_path="synthetic_span_audit.csv",
    n_per_category=25,
    seed=13,
)

Saved 100 audit rows to synthetic_span_audit.csv
category
Coherence            25
Format               25
Lacks synthesis      25
Unsupported claim    25
Name: count, dtype: int64


In [5]:
import json
from pathlib import Path
from itertools import product, combinations

import pandas as pd


AUDIT_COLS = ["category_valid", "boundary_plausible"]


def load_audit_file(path):
    """
    Loads either a CSV audit file or a JSON audit file.
    JSON can be either:
      1. a list of rows
      2. a dict grouped by category
    """
    path = Path(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    rows = []

    if isinstance(data, list):
        rows = data

    elif isinstance(data, dict):
        for category, examples in data.items():
            for ex in examples:
                base = {
                    "category": category,
                    "document": ex.get("document", ""),
                    "reason": ex.get("reason", ""),
                }

                # If audit labels are stored at example level
                if all(col in ex for col in AUDIT_COLS):
                    rows.append({**base, **{col: ex.get(col) for col in AUDIT_COLS}})
                    continue

                # If audit labels are stored at span level
                for span in ex.get("spans", []):
                    row = {
                        **base,
                        "span": span.get("span", span.get("text", "")),
                    }
                    for col in AUDIT_COLS:
                        row[col] = span.get(col, ex.get(col, None))
                    rows.append(row)

    else:
        raise ValueError("Unsupported file format.")

    return pd.DataFrame(rows)


def clean_binary_columns(df, cols=AUDIT_COLS):
    """
    Converts audit columns to 0/1 integers.
    Accepts values like 1, 0, '1', '0', True, False.
    """
    df = df.copy()

    for col in cols:
        df[col] = df[col].map({
            1: 1, 0: 0,
            "1": 1, "0": 0,
            True: 1, False: 0,
            "true": 1, "false": 0,
            "True": 1, "False": 0,
            "yes": 1, "no": 0,
            "Yes": 1, "No": 0,
        })

    missing = df[cols].isna().sum()
    if missing.any():
        print("Warning: missing/non-binary values found:")
        print(missing[missing > 0])

    df = df.dropna(subset=cols)
    df[cols] = df[cols].astype(int)

    return df


def summarize_audit(df, cols=AUDIT_COLS):
    """
    Computes:
      1. percentage valid for each individual aspect
      2. percentage valid for each inclusive combination
      3. exact 000--111 combination distribution
      4. all-three-valid percentage
    """
    df = clean_binary_columns(df, cols)
    n = len(df)

    if n == 0:
        raise ValueError("No valid audited rows found.")

    # 1. Individual aspect percentages
    individual = (
        df[cols].mean()
        .mul(100)
        .round(2)
        .rename("percent_1")
        .reset_index()
        .rename(columns={"index": "aspect"})
    )

    # 2. Inclusive combinations, e.g. category_valid & boundary_plausible
    inclusive_rows = []
    for r in range(1, len(cols) + 1):
        for subset in combinations(cols, r):
            mask = df[list(subset)].eq(1).all(axis=1)
            inclusive_rows.append({
                "combination": " + ".join(subset),
                "count": int(mask.sum()),
                "percent": round(mask.mean() * 100, 2),
            })

    inclusive = pd.DataFrame(inclusive_rows)

    # 3. Exact combination distribution, e.g. exactly 1,1,0
    exact_rows = []
    for values in product([0, 1], repeat=len(cols)):
        mask = (df[cols] == list(values)).all(axis=1)
        exact_rows.append({
            **dict(zip(cols, values)),
            "pattern": "".join(map(str, values)),
            "count": int(mask.sum()),
            "percent": round(mask.mean() * 100, 2),
        })

    exact = pd.DataFrame(exact_rows).sort_values("pattern")

    # 4. All three valid
    all_three_mask = df[cols].eq(1).all(axis=1)
    all_three = {
        "count": int(all_three_mask.sum()),
        "total": int(n),
        "percent": round(all_three_mask.mean() * 100, 2),
    }

    return {
        "n": n,
        "all_three": all_three,
        "individual": individual,
        "inclusive_combinations": inclusive,
        "exact_combinations": exact,
    }


def summarize_audit_by_category(df, category_col="category", cols=AUDIT_COLS):
    """
    Optional: same summaries per category.
    """
    df = clean_binary_columns(df, cols)

    summaries = {}
    for category, group in df.groupby(category_col):
        summaries[category] = summarize_audit(group, cols)

    return summaries

In [6]:
df = pd.read_csv("synthetic_span_audit_annotated.csv")

by_cat = summarize_audit_by_category(df)

for category, summary in by_cat.items():
    print("\n", category)
    print("All three:", summary["all_three"])
    print(summary["exact_combinations"])


 Coherence
All three: {'count': 16, 'total': 25, 'percent': np.float64(64.0)}
   category_valid  boundary_plausible pattern  count  percent
0               0                   0      00      6     24.0
1               0                   1      01      1      4.0
2               1                   0      10      2      8.0
3               1                   1      11     16     64.0

 Format
All three: {'count': 21, 'total': 25, 'percent': np.float64(84.0)}
   category_valid  boundary_plausible pattern  count  percent
0               0                   0      00      3     12.0
1               0                   1      01      0      0.0
2               1                   0      10      1      4.0
3               1                   1      11     21     84.0

 Lacks synthesis
All three: {'count': 21, 'total': 25, 'percent': np.float64(84.0)}
   category_valid  boundary_plausible pattern  count  percent
0               0                   0      00      2      8.0
1               